# A cascade of pretrained YOLO models on a photographic collection: the Fonds Thierry

Computer Vision & Heritage, session 1 (AI for heritage, images as matrices), hands-on at the end of part 2

Goal: what generic, supervised models (COCO and ADE20K: closed vocabularies; depth) tell an institution about a collection, step after step.

Images: INHA, Fonds Nicole et Jean-Michel Thierry, licence CC BY-NC-ND 4.0. ND: do not share the derived images you make (crop, mask, cut-out, annotated or blurred image, gallery).

In [ ]:
import os
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from PIL import Image, ImageDraw, ImageFilter, ImageStat
from ultralytics import YOLO

## 0. Paths and models

In [ ]:
# The 8 scans
DATASET = os.path.join("..", "datasets", "dataset-s01-tp")

In [ ]:
# Polars tables: up to 20 rows shown, strings up to 50 characters
pl.Config.set_tbl_rows(20).set_fmt_str_lengths(50)
# Tables written by the notebook (data/: not versioned)
OUT = os.path.join(DATASET, "..", "..", "data", "thierry")
os.makedirs(OUT, exist_ok=True)

In [ ]:
# The four models; first run: each downloads its weights into this folder
# Object detection, COCO, 80 classes (5.5 MB)
det_model = YOLO("yolo26n.pt")
# Instance segmentation, COCO, 80 classes: a box and a mask per object (6.7 MB)
seg_model = YOLO("yolo26n-seg.pt")
# Semantic segmentation, ADE20K, 150 classes: a class per pixel (3.5 MB)
sem_model = YOLO("yolo26n-sem-ade20k.pt")
# Depth estimation: a distance per pixel, in metres (13 MB)
depth_model = YOLO("yolo26n-depth.pt")

## 1. The collection

List the `.jpg` files of `DATASET`, in alphabetical order: `files` (`os.listdir`, `sorted`).

Give each slide a short name, `slides`: the file name without the beginning shared by all the files (`os.path.commonprefix`) and without the extension (`removeprefix`, `removesuffix`).

Display `slides`.

Show the 8 scans in a grid of 2 × 4 axes (`plt.subplots`), each titled with its short name, without axes: four 35 mm slides in their mounts, four medium-format films with their black edges.

Open the first scan (`Image.open`): `village`, a 35 mm slide in its mount, captioned "URANC' village".

Print `village`: its class, mode and size (printed, so that the 15-megapixel scan is not embedded in the notebook).

Display its size: PIL gives (width, height).

Display its mode.

Display the shape of the same image as a NumPy array (`np.asarray`): (height, width, channels), rows first.

## 2. Remove the mount, without a model

Strategy: (1) the colour of the mount, measured near the four corners; (2) threshold: the pixels close to this colour; (3) keep the zone: the columns, then the rows, where such pixels are less than half.

Convert `village` to a NumPy array of integers (`astype(int)`): `arr`, of shape (H, W, 3).

(1) Write `mount_colour(arr)`: the median (R, G, B) of the pixels of four square patches near the corners, from `p0 = int(0.04 * min(W, H))` to `p1 = int(0.12 * min(W, H))` pixels from each corner (the patches reshaped to (N, 3), concatenated, then `np.median` along axis 0).

Apply it to `arr`: `mount`.

Display `mount`.

(2) Threshold: compute `like_mount`, True where the sum over R, G, B of the absolute differences between the pixel and `mount` is under 25.

Display its shape: one boolean per pixel, (H, W).

Show `like_mount` as an image (`plt.imshow`, `cmap="gray"`): the pixels like the mount in white.

(3) Compute the share of the pixels like the mount in each column (x): `column_share`.

… and in each row (y): `row_share`.

Plot `column_share` and `row_share` side by side, each with a horizontal line at 0.5 (`axhline`): the columns and the rows of the photo are under 0.5.

Write `longest_run(flags)`: `(start, end)` of the longest run of consecutive `True` in a sequence of booleans, `end` excluded. Then `keep_zone(like_mount, share=0.5, passes=3)`: start from all the rows (`y1, y2 = 0, H`) and, `passes` times, take as columns `x1, x2` the longest run of columns whose share over the rows `y1:y2` is under `share`, then as rows `y1, y2` the longest run of rows whose share over the columns `x1:x2` is under `share`; it returns `(x1, y1, x2, y2)`.

Apply it to `like_mount`: `box`, the zone kept, (x1, y1, x2, y2) in pixels of the scan.

Display `box`.

Write `frame_box(scan)`, the whole strategy for any scan (threshold 25, share 0.5; both set on these 8 scans: check the frames found), and the format from the shape of the zone, `ratio = (x2 - x1) / (y2 - y1)`: `"film"` if 0.85 < ratio < 1.15 (square, medium-format 6 × 6), else `"35mm-portrait"` if ratio < 1, else `"35mm-landscape"` (2:3, 35 mm slide). It returns `(format, (x1, y1, x2, y2))`.

Apply it to `village`.

Crop the scan to its frame (`crop`): `photo`.

Display the size of `photo`, (width, height).

Detect the objects (COCO) on the whole scan with `det_model` (`predict`, `verbose=False`, first result): `r_scan`.

Count the classes detected (`Counter` of the class names: `r_scan.names`, `r_scan.boxes.cls`).

Do the same on the photo alone: `r_photo` (the network sees images of 640 pixels on their longest side: on the photo, people are larger).

Count the classes detected on the photo.

Apply `frame_box` to the 8 scans: `frames` (slide → (format, frame)) and `photos` (slide → the scan cropped to its frame); print one line per slide: name, format, frame.

Show the 8 scans in a grid of 2 × 4 axes with the frame found drawn in red (`plt.Rectangle`, `fill=False`), titled with the name and the format.

## 3. Orientation check: where is the sky? (semantic segmentation, ADE20K)

Segment `photo` with `sem_model` (first result): `r`.

Display `r.semantic_mask`: one class index per pixel, (H, W), the size of the photo.

Convert it to a NumPy array (`data.cpu().numpy()`): `sem`.

Read the index of the class "sky" from `r.names`: `sky`.

Find the row index of each sky pixel (`np.nonzero`): `sky_rows`.

Compute the vertical centre of the sky, as a fraction of the height: 0 = top, 1 = bottom.

Show the sky as an image (`sem == sky`, white).

Write the same as functions: `semantic_map(photo)`, the class of each pixel of a photo (a NumPy array); `sky_centre(sem)`, the vertical centre of its sky, rounded to 3 decimals, or `None` without sky.

Print the sky centre of `photo` turned 180° (`rotate`): no sky, or sky at the bottom.

Compute on the 8 photos `sems` (slide → class map) and `upright` (slide → `True` if there is sky and its centre is in the upper half; otherwise `False`, to check visually); print one line per slide: the sky centre, the sky centre of the photo turned 180°, `upright`.

## 4. Scene description: share of the pixels per class (ADE20K)

List the classes present in `sem` and their number of pixels (`np.unique`, `return_counts=True`): `ids`, `n_pixels`.

Display the three largest (`np.argsort`): a list of (class name from `r.names`, share of the pixels in %, rounded to 0.1).

Write the same as a function, `scene(sem)`: the three largest classes of a class map with their shares, as one string, `"<class> <share> %"` joined by commas (`sem_model.names`: the same dict as `r.names`).

Apply it to the 8 photos: `scenes` (slide → string); print one line per slide.

## 5. Object inventory (COCO detection): a table

Turn the detections on the photo (`r_photo`, step 2) into a table, one row per box, the corners as fractions of the photo size, 3 decimals (`to_df`, `normalize=True`, `decimals=3`): `df`.

Display `df`.

Display its type.

Display its columns and their types (`schema`): `box` is a struct of four numbers.

Put the detections of the 8 photos in one table, `objects`: the table of each photo with the name of the slide as a new column (`with_columns`, `pl.lit`), concatenated with `pl.concat(..., how="diagonal")` (a photo without any object gives an empty table).

Display its first rows (`head`).

Split the column `box` into one column per corner (`unnest`): `objects`.

Display its first rows.

Count the objects per slide and class (`group_by`, `len`), sorted by slide and class.

Keep everything but persons (`filter`): `others`. Which ones are right?

Display `others`.

Show each of them, cropped from its photo with its own box (fractions → pixels: times the width or the height of the photo), titled with its class and confidence.

Write `objects` to `objects.csv` in `OUT` (`write_csv`).

Display the first rows of the same table in pandas (`to_pandas`).

## 6. People: masks (instance segmentation, COCO) and depth

Estimate the depth of `photo` with `depth_model` (first result): `r`.

Display `r.depth`: one distance per pixel, in metres, (H, W), the size of the photo.

Convert it to a NumPy array: `depth`.

Display the nearest estimate on the photo (m).

Display the farthest estimate on the photo (m).

Show the depth map (`cmap="magma_r"`) with a colour bar in metres (`plt.colorbar`).

Segment the instances of `photo` with `seg_model` (first result): `r`, a box and a mask per object.

Read the index of the class "person" from `r.names`: `person`.

Keep only the persons: `people` (indexing a `Results` object with a condition on `r.boxes.cls` keeps the boxes and masks selected).

Display the shape of the outline of the first person, `people.masks.xy[0]`, in pixels of the photo: (number of points, 2).

Display its first 5 points.

Write `polygon_mask(xy, size)`: a black image of the given size (`Image.new`, mode `"L"`), the outline filled in white (`ImageDraw.Draw(...).polygon`, `fill=255`); and `median_depth(mask, depth)`: the median depth (m) of the pixels inside a mask, as a `float`.

Build the mask of the first person, at the size of the photo: `mask`.

Show `mask`.

Compute its distance (m): the median depth of its pixels; background beyond 5 m.

For each person of the village photo, build its mask and its cut-out: the photo made transparent outside the mask (`copy`, `putalpha`) and cropped to its box (`people.boxes.xyxy`): two lists, `masks` and `cutouts`.

Show the cut-outs side by side in `gallery`, a white image 300 pixels high (`Image.new`): each brought to 300 pixels high (`resize`) and pasted with its transparency as mask (`paste`).

Repeat these steps on the 8 photos: a list `rows`, one dict per person: `slide`, `conf` (2 decimals), `box` (a dict `x1`, `y1`, `x2`, `y2`, rounded), `height` (px), `depth` (median depth, m, 2 decimals), `plane` (`"background"` beyond 5 m, `"foreground"` otherwise); skip a photo where `r.masks` is `None`.

Make a Polars DataFrame of `rows`: `persons`.

Display `persons`.

Count the persons per slide and plane, sorted.

Check on the village photo (`slides[0]`): add the column `height_x_depth`, apparent height (px) × depth (m), and show it with `height`, `depth` and `plane`, from near to far: same order of magnitude for adults standing, higher for the far persons.

## 7. Option: blur the people

Merge the masks of the village photo into `union`, white wherever there is a person (pixel-wise maximum: `np.maximum.reduce`, then `Image.fromarray`).

Show the photo blurred (Gaussian blur, radius 25 pixels: `ImageFilter.GaussianBlur`) where `union` is white, unchanged elsewhere (`Image.composite`).

## 8. Colour cast: mean of each channel

Display the means of R, G and B over `photo` (`ImageStat.Stat`).

Compute them on the 8 photos: a list `means`, one dict per slide: `slide`, `R`, `G`, `B` (rounded).

Make a Polars DataFrame of `means`: `colours`.

Display `colours`.

## 9. Report: one row per slide

Count the persons and the persons in the background per slide (`persons`, step 6; `group_by`, `agg`): `people_counts`, columns `persons` and `background`.

Display it, sorted by slide.

Count the boats and the cars per slide (`objects`, step 5): `object_counts`, columns `boats` and `cars`.

Display it, sorted by slide.

Build `report`, one row per slide: `slide`, `format` and `frame` (step 2; the frame as a string), `upright` (3), `scene` (4), then the counts and the mean colour (8) joined on `slide` (`join`, `how="left"`); a slide without any person or object: 0 rather than null (`fill_null`); sorted by slide.

Display `report`.

Write it to `report.csv` in `OUT`.

Print the file.